In [3]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# CatBoost

In [14]:
blocks = gpd.read_parquet('../data/traning_data/prepared_blocks.parquet')
blocks.head()

,geometry,residential,business,recreation,industrial,transport,special,agriculture,land_use,share,...,cluster,fsi,gsi,mxi,l,osr,share_living,share_non_living,morphotype,area_accessibility
0,"POLYGON ((352083.617 6633950.146, 352240.448 6...",0.099000,0.0,0.079912,0.000000,0.401072,0.0,0.417018,AGRICULTURE,0.417018,...,3.0,0.000503,0.000503,0.000000,1.000000,1985.451134,0.000000,1.000000,low-rise non-residential,113.847998
1,"POLYGON ((346700.642 6618453.176, 346681.107 6...",1.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,RESIDENTIAL,1.000000,...,2.0,0.064120,0.061465,0.687302,1.043202,14.637095,0.716995,0.326207,individual residential,145.884534
2,"POLYGON ((347043.363 6618261.219, 347042.608 6...",0.729125,0.0,0.270875,0.000000,0.000000,0.0,0.000000,RESIDENTIAL,0.729125,...,2.0,0.034748,0.033472,0.693362,1.038117,27.815100,0.719791,0.318326,individual residential,148.543184
3,"POLYGON ((354879.039 6618859.116, 354845.405 6...",0.454375,0.0,0.000000,0.000000,0.144935,0.0,0.399984,RESIDENTIAL,0.454375,...,2.0,0.183930,0.078846,0.667715,2.332762,5.008184,1.557620,0.775142,low-rise model,132.691062
4,"POLYGON ((347215.933 6646341.789, 347245.429 6...",0.108707,0.0,0.000000,0.767131,0.057528,0.0,0.000000,INDUSTRIAL,0.767131,...,5.0,1.000310,0.250078,0.000000,4.000000,0.749690,0.000000,4.000000,mid-rise non-residential,99.160727


In [ ]:
import numpy as np
import pandas as pd

from tqdm import tqdm
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold, ParameterSampler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from libpysal.weights import DistanceBand, lag_spatial
from esda.moran import Moran
from libpysal.weights import Queen

from catboost import CatBoostRegressor, Pool
import shap

# =========================
# 0) Конфиг
# =========================
feature_cols = [
    'residential','business','recreation','industrial','transport','special',
    'agriculture','land_use','share','footprint_area','build_floor_area',
    'living_area','non_living_area','population','site_area','fsi','gsi',
    'mxi','l','morphotype','area_accessibility'
]
cat_features = ['land_use', 'morphotype']
numeric_feats = [c for c in feature_cols if c not in cat_features]

target_col = 'log_total_price'
radius_list = [300, 500, 1000, 2000, 3000]

# =========================
# 1) Служебные функции
# =========================
def build_radii_weights(df, radii):
    w = {}
    for r in radii:
        wr = DistanceBand.from_dataframe(df, threshold=r, binary=True, silence_warnings=True)
        wr.transform = 'r'
        w[r] = wr
    return w

def build_lags(df, radii, numeric_cols):
    df = df.copy()
    w_r = build_radii_weights(df, radii)
    for feat in numeric_cols:
        vec = df[feat].fillna(df[feat].mean())
        for r, w in w_r.items():
            df[f'lag{r}_{feat}'] = lag_spatial(w, vec)
    for r, w in w_r.items():
        df[f'n_neighbors_{r}'] = pd.Series({i: len(nb) for i, nb in w.neighbors.items()})
    return df

def prep_cat(df, cat_cols):
    df = df.copy()
    for c in cat_cols:
        df[c] = df[c].astype('string').fillna('missing')
    return df

def feature_names(df, base_cols):
    extra = [c for c in df.columns if c.startswith('lag') or c.startswith('n_neighbors_')]
    final = list(dict.fromkeys(base_cols + extra))
    final = [c for c in final if c != target_col]
    return final

def pools_from_frames(X_tr, y_tr, X_te, y_te, cat_cols, feats):
    tr_pool = Pool(X_tr[feats], label=y_tr, cat_features=cat_cols, feature_names=feats)
    te_pool = Pool(X_te[feats], label=y_te, cat_features=cat_cols, feature_names=feats)
    return tr_pool, te_pool

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

# =========================
# 2) Группы для пространственного сплита
# =========================
if {'x','y'}.issubset(blocks.columns):
    coords = blocks[['x','y']].to_numpy()
else:
    cent = blocks.geometry.centroid
    coords = np.column_stack([cent.x.values, cent.y.values])

groups = KMeans(n_clusters=10, random_state=42, n_init='auto').fit_predict(coords)

# =========================
# 3) Внешний честный тест-фолд
# =========================
gkf_outer = GroupKFold(n_splits=5)
outer_splits = list(gkf_outer.split(blocks, blocks[target_col], groups))
train_idx, test_idx = outer_splits[0]

df_train = blocks.iloc[train_idx].copy()
df_test  = blocks.iloc[test_idx].copy()

# =========================
# 4) Подбор гиперпараметров с внутренними пространственными фолдами + логи
# =========================
param_grid = {
    'depth': [5,6,7,8],
    'learning_rate': [0.02, 0.03, 0.05],
    'l2_leaf_reg': [3,6,9,12],
    'random_strength': [0.5,1.0,1.5,2.0],
    'bagging_temperature': [0.25, 0.5, 1.0, 2.0]  # для bootstrap_type='Bayesian'
}
param_iter = list(ParameterSampler(param_grid, n_iter=24, random_state=42))

best_params = None
best_cv = np.inf

inner_groups = groups[train_idx]
gkf_inner = GroupKFold(n_splits=5)

print("=== HPO: старт подбора параметров ===")
for params in tqdm(param_iter, desc="Trials"):
    fold_rmses = []
    for k, (tr, va) in enumerate(gkf_inner.split(df_train, df_train[target_col], inner_groups), 1):
        tr_df = df_train.iloc[tr].copy()
        va_df = df_train.iloc[va].copy()

        tr_df = build_lags(tr_df, radius_list, numeric_feats)
        va_df = build_lags(va_df, radius_list, numeric_feats)

        tr_df = prep_cat(tr_df, cat_features)
        va_df = prep_cat(va_df, cat_features)

        feats = feature_names(tr_df, feature_cols)

        tr_pool, va_pool = pools_from_frames(
            tr_df, tr_df[target_col], va_df, va_df[target_col], cat_features, feats
        )

        model = CatBoostRegressor(
            loss_function='RMSE',
            eval_metric='RMSE',
            iterations=15000,
            od_type='Iter', od_wait=300,
            bootstrap_type='Bayesian',
            grow_policy='SymmetricTree',
            random_seed=42,
            verbose=500,                 # печать каждые 200 итераций
            **params
        )

        print(f"\n[trial {params}] fold={k}")
        model.fit(tr_pool, eval_set=va_pool, early_stopping_rounds=300, use_best_model=True)

        er = model.get_evals_result()
        best_it = model.get_best_iteration() or model.tree_count_
        tr_rmse_last = er['learn']['RMSE'][best_it-1]
        va_rmse_last = er['validation']['RMSE'][best_it-1]
        print(f"best_it={best_it}  RMSE learn={tr_rmse_last:.4f}  val={va_rmse_last:.4f}")

        pred = model.predict(va_df[feats])
        fold_rmses.append(rmse(va_df[target_col], pred))

    cv_rmse = float(np.mean(fold_rmses))
    print(f"[trial end] params={params}  CV_RMSE={cv_rmse:.4f}")
    if cv_rmse < best_cv:
        best_cv = cv_rmse
        best_params = params
        print(f"--> NEW BEST: {best_params}  CV_RMSE={best_cv:.4f}")

print("Лучшие параметры:", best_params, "CV_RMSE:", round(best_cv, 4))

# =========================
# 5) Финальная модель: тренируем на train, валидируем на внешнем test + логи
# =========================
df_train_lag = build_lags(df_train, radius_list, numeric_feats)
df_test_lag  = build_lags(df_test,  radius_list, numeric_feats)

df_train_lag = prep_cat(df_train_lag, cat_features)
df_test_lag  = prep_cat(df_test_lag,  cat_features)

feats_final = feature_names(df_train_lag, feature_cols)

train_pool = Pool(df_train_lag[feats_final], label=df_train_lag[target_col],
                  cat_features=cat_features, feature_names=feats_final)
test_pool  = Pool(df_test_lag[feats_final],  label=df_test_lag[target_col],
                  cat_features=cat_features, feature_names=feats_final)

final_model = CatBoostRegressor(
    loss_function='RMSE', eval_metric='RMSE',
    iterations=20000, od_type='Iter', od_wait=400,
    bootstrap_type='Bayesian', grow_policy='SymmetricTree',
    random_seed=42,
    verbose=500,               
    **best_params
)
print("\n=== Финальная модель: обучение ===")
final_model.fit(train_pool, eval_set=test_pool, early_stopping_rounds=400, use_best_model=True)

er = final_model.get_evals_result()
best_it = final_model.get_best_iteration() or final_model.tree_count_
print(f"[final] best_it={best_it}  learn_RMSE={er['learn']['RMSE'][best_it-1]:.4f}  "
      f"test_RMSE={er['validation']['RMSE'][best_it-1]:.4f}")

# =========================
# 6) Оценка на внешнем тесте
# =========================
y_true = df_test_lag[target_col].values
y_pred = final_model.predict(df_test_lag[feats_final])

print(f"R²   = {r2_score(y_true, y_pred):.4f}")
print(f"MAE  = {mean_absolute_error(y_true, y_pred):.4f}")
print(f"RMSE = {rmse(y_true, y_pred):.4f}")

# =========================
# 7) Проверка пространственной автокорреляции остатков
# =========================
resid = y_true - y_pred
wq = Queen.from_dataframe(df_test_lag)  # нужна geometry
wq.transform = 'r'
mi = Moran(resid, wq)
print(f"Moran's I: {mi.I:.4f}, p≈{mi.p_sim:.4f}")

# =========================
# 8) SHAP на сэмпле
# =========================
sample = df_test_lag[feats_final].sample(n=min(1500, len(df_test_lag)), random_state=42)

explainer = shap.TreeExplainer(final_model)  # feature_perturbation не нужен
shap_vals = explainer.shap_values(sample, check_additivity=False)  # без approximate

shap.summary_plot(shap_vals, sample, plot_type="bar")
shap.summary_plot(shap_vals, sample)


In [ ]:
# final_model.save_model('./data/catboost_model_4_12.cbm')